# Multi-Agent Systems & AutoGen

**WatSPEED Agentic AI prep — Week 5-6 - multi-agent**

Runs offline. Set `OPENAI_API_KEY` to swap the stub model for a real one.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / 'agentkit.py').exists()
                      else pathlib.Path.cwd() / 'notebooks'))
from agentkit import *

## Multi-agent systems

The syllabus lists **AutoGen** and "autonomous multi-agent systems design". The premise:
several narrow agents with distinct roles and tools often beat one agent with a long
prompt — because each has a smaller job, a smaller context, and a clearer failure mode.

Three orchestration patterns cover most real systems:

1. **Sequential (pipeline)** — analyst -> reviewer -> writer.
2. **Group chat with a manager** — a router picks who speaks next.
3. **Hand-off** — an agent transfers control when the task leaves its remit.

The cost is real: more agents means more calls, more latency, and more places to lose
information at a boundary.

In [2]:
from dataclasses import dataclass, field

@dataclass
class Agent:
    name: str
    role: str
    llm: object
    tools: ToolBox | None = None
    received: list = field(default_factory=list)

    def act(self, message: str) -> str:
        self.received.append(message)
        prompt = [{"role": "system", "content": self.role}, {"role": "user", "content": message}]
        reply = self.llm.chat(prompt, tools=self.tools.schemas() if self.tools else None)
        if reply.wants_tool and self.tools:
            call = reply.tool_calls[0]
            result = self.tools.call(call["name"], call["arguments"])
            return f"[{self.name} used {call['name']}] {result}"
        return reply.content

### Pattern 1 — sequential pipeline

In [3]:
stats = ToolBox()

@stats.tool("Chi-square test of independence", schema(row_var='string', col_var='string'))
def chi_square(row_var: str, col_var: str) -> dict:
    return {"chi2": 12.41, "dof": 3, "p": 0.006, "n": 2041, "weighted": False}

analyst = Agent("Analyst", "You run statistical tests.",
                get_llm([LLMResponse(tool_calls=[{"name": "chi_square",
                        "arguments": {"row_var": "age_group", "col_var": "trusts_ai"}}])], verbose=False),
                tools=stats)

reviewer = Agent("Reviewer", "You check methodology. Be sceptical.",
                 get_llm([LLMResponse(content="REJECT: result is unweighted; "
                                              "survey weights are required before reporting.")], verbose=False))

writer = Agent("Writer", "You write plain-language summaries.",
               get_llm([LLMResponse(content="Older respondents report higher trust in AI "
                                            "(chi2=12.41, p=0.006, N=2041, weighted).")], verbose=False))

banner("Sequential pipeline")
msg = "Test whether AI trust varies by age group."
for step, agent in enumerate([analyst, reviewer, writer], start=1):
    msg = agent.act(msg)
    trace(step, agent.name, msg)


Sequential pipeline
  [ 1] Analyst      | [Analyst used chi_square] {'chi2': 12.41, 'dof': 3, 'p': 0.006, 'n': 2041, 'weighted': False}
  [ 2] Reviewer     | REJECT: result is unweighted; survey weights are required before reporting.
  [ 3] Writer       | Older respondents report higher trust in AI (chi2=12.41, p=0.006, N=2041, weighted).


### Why the reviewer matters

Notice the reviewer caught the unweighted estimate. A single agent asked to
"analyse and check your work" tends not to — the same context that produced the error
is being used to judge it. **Separate context is the mechanism**, not the extra prompt.

### Pattern 2 — group chat with a manager

In [4]:
class GroupChat:
    """A manager routes each turn. AutoGen's GroupChatManager, minus the machinery."""

    def __init__(self, agents: list[Agent], manager_llm):
        self.agents = {a.name: a for a in agents}
        self.manager = manager_llm
        self.transcript: list[tuple[str, str]] = []

    def run(self, task: str, max_turns: int = 5) -> list:
        message = task
        for turn in range(1, max_turns + 1):
            choice = self.manager.chat([
                {"role": "system", "content": f"Pick the next speaker from {list(self.agents)}, "
                                              "or reply DONE."},
                {"role": "user", "content": message},
            ]).content.strip()

            if choice == "DONE" or choice not in self.agents:
                trace(turn, "manager", f"DONE ({choice})")
                break
            trace(turn, "manager", f"-> {choice}")
            message = self.agents[choice].act(message)
            trace(turn, choice, message)
            self.transcript.append((choice, message))
        return self.transcript

manager = get_llm([LLMResponse(content="Analyst"), LLMResponse(content="Reviewer"),
                   LLMResponse(content="Writer"), LLMResponse(content="DONE")], verbose=False)

for a in (analyst, reviewer, writer):
    a.llm.reset() if hasattr(a.llm, "reset") else None

banner("Group chat")
chat = GroupChat([analyst, reviewer, writer], manager)
_ = chat.run("Test whether AI trust varies by age group and summarise it.")


Group chat
  [ 1] manager      | -> Analyst
  [ 1] Analyst      | [Analyst used chi_square] {'chi2': 12.41, 'dof': 3, 'p': 0.006, 'n': 2041, 'weighted': False}
  [ 2] manager      | -> Reviewer
  [ 2] Reviewer     | REJECT: result is unweighted; survey weights are required before reporting.
  [ 3] manager      | -> Writer
  [ 3] Writer       | Older respondents report higher trust in AI (chi2=12.41, p=0.006, N=2041, weighted).
  [ 4] manager      | DONE (DONE)


### Pattern 3 — hand-off

An agent decides the task isn't its job and transfers, carrying a summary. The summary
is the risky part: everything not in it is lost at the boundary.

In [5]:
def handoff(from_agent: str, to_agent: str, summary: str) -> dict:
    return {"type": "handoff", "from": from_agent, "to": to_agent, "context": summary}

h = handoff("Analyst", "Methodologist",
            "chi2=12.41 p=0.006 N=2041 on age_group x trusts_ai; weights NOT applied")
show("Hand-off payload", h)
print("\nEverything the Methodologist knows is in 'context'. "
      "Anything the Analyst saw but did not write down is gone.")

Hand-off payload:
  {
    "type": "handoff",
    "from": "Analyst",
    "to": "Methodologist",
    "context": "chi2=12.41 p=0.006 N=2041 on age_group x trusts_ai; weights NOT applied"
  }

Everything the Methodologist knows is in 'context'. Anything the Analyst saw but did not write down is gone.


### The AutoGen syntax

```python
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

analyst  = AssistantAgent("Analyst",  system_message="You run statistical tests.")
reviewer = AssistantAgent("Reviewer", system_message="You check methodology.")
user     = UserProxyAgent("User", human_input_mode="NEVER", code_execution_config=False)

chat = GroupChat(agents=[user, analyst, reviewer], messages=[], max_round=6)
user.initiate_chat(GroupChatManager(groupchat=chat), message="Test AI trust by age group.")
```

`AssistantAgent` is the `Agent` class above; `GroupChatManager` is the routing loop.

---
### Try it yourself

1. Make the reviewer send work *back* to the analyst instead of forward. What stops a loop?
2. Give each agent its own `ToolBox`. Which tools must the reviewer be denied, and why?
3. Time it: how many model calls did the pipeline use versus one agent? When is that worth it?